In [1]:
import sys
print("Python:", sys.version)
print("Executable:", sys.executable)

import numpy as np
import lightgbm
import shap

print("lightgbm:", lightgbm.__version__)
print("shap:", shap.__version__)
print("OK imports")


Python: 3.10.13 (main, Sep 11 2023, 13:44:35) [GCC 11.2.0]
Executable: /opt/conda/bin/python


/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


lightgbm: 4.6.0
shap: 0.49.1
OK imports


In [2]:
import sys
!{sys.executable} -m pip install -U pip
!{sys.executable} -m pip install lightgbm shap numpy pandas scikit-learn matplotlib


In [3]:
import lightgbm, shap
print(lightgbm.__version__, shap.__version__)


4.6.0 0.49.1


In [4]:
import lightgbm, shap
print("lightgbm:", lightgbm.__version__)
print("shap:", shap.__version__)


lightgbm: 4.6.0
shap: 0.49.1


In [7]:
import os
MODEL_FILE = "/workspace/ember2024_assets/models/EMBER2024_PE.model"
print("exists:", os.path.exists(MODEL_FILE), "size:", os.path.getsize(MODEL_FILE))


exists: True size: 3756042


In [8]:
import lightgbm as lgb

bst = lgb.Booster(model_file=MODEL_FILE)
print("Loaded baseline model. Trees:", bst.num_trees())



Loaded baseline model. Trees: 500


In [10]:
import os, thrember

DATA_DIR = "/workspace/ember2024_assets/data"
os.makedirs(DATA_DIR, exist_ok=True)

# download both splits
thrember.download_dataset(DATA_DIR, file_type="PE", split="train")
thrember.download_dataset(DATA_DIR, file_type="PE", split="test")

# now vectorize will succeed
thrember.create_vectorized_features(DATA_DIR)

X, y = thrember.read_vectorized_features(DATA_DIR, subset="test")
print("X:", X.shape, "y:", y.shape)


Unzipping...
Unzipping...
Unzipping...
Unzipping...
Unzipping...
Unzipping...
Preparing to vectorize raw features
Vectorizing training set


100%|█████████████████████████| 4680000/4680000 [22:26<00:00, 3475.34it/s]


Vectorizing test set


100%|█████████████████████████| 1080000/1080000 [05:21<00:00, 3355.36it/s]


Vectorizing challenge set


ValueError: Did not find any .jsonl files matching criteria

In [11]:
DATA_DIR = "/workspace/ember2024_assets/data"

thrember.download_dataset(DATA_DIR, file_type="PE", split="challenge")
thrember.create_vectorized_features(DATA_DIR)  # now it will finish

X, y = thrember.read_vectorized_features(DATA_DIR, subset="test")
print("X:", X.shape, "y:", y.shape)


Preparing to vectorize raw features
Vectorizing training set


100%|█████████████████████████| 4680000/4680000 [25:18<00:00, 3081.72it/s]


Vectorizing test set


100%|█████████████████████████| 1080000/1080000 [04:58<00:00, 3619.42it/s]


Vectorizing challenge set


100%|███████████████████████████████| 6315/6315 [00:02<00:00, 2491.02it/s]


X: (1080000, 2568) y: (1080000,)


In [12]:
import os
for f in ["X_train.dat","y_train.dat","X_test.dat","y_test.dat"]:
    p = os.path.join("/workspace/ember2024_assets/data", f)
    print(f, "exists:", os.path.exists(p), "sizeMB:", (os.path.getsize(p)/1024/1024 if os.path.exists(p) else None))


X_train.dat exists: True sizeMB: 45845.947265625
y_train.dat exists: True sizeMB: 17.852783203125
X_test.dat exists: True sizeMB: 10579.833984375
y_test.dat exists: True sizeMB: 4.119873046875


In [13]:
X, y = thrember.read_vectorized_features("/workspace/ember2024_assets/data", subset="test")
print("X:", X.shape, "y:", y.shape)


X: (1080000, 2568) y: (1080000,)


In [15]:
import os
import numpy as np
import shap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR = "/workspace/ember2024_assets/shap_out"
os.makedirs(OUT_DIR, exist_ok=True)

rng = np.random.default_rng(42)
idx = rng.choice(X.shape[0], size=min(5000, X.shape[0]), replace=False)
X_small = X[idx]

bg_idx = rng.choice(X_small.shape[0], size=min(1000, X_small.shape[0]), replace=False)
X_bg = X_small[bg_idx]

explainer = shap.TreeExplainer(bst, data=X_bg, feature_perturbation="interventional")
sv = explainer.shap_values(X_small, check_additivity=False)

shap_vals = sv[1] if isinstance(sv, list) else sv

# Summary plot
plt.figure()
shap.summary_plot(shap_vals, X_small, show=False, max_display=30)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "summary_top30.png"), dpi=200)
plt.close()

# Bar plot
plt.figure()
shap.summary_plot(shap_vals, X_small, plot_type="bar", show=False, max_display=30)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "importance_bar_top30.png"), dpi=200)
plt.close()

print("Saved SHAP plots in:", OUT_DIR)


100%|===================| 4984/5000 [02:50<00:00]        

Saved SHAP plots in: /workspace/ember2024_assets/shap_out


In [16]:
import glob
glob.glob("/workspace/ember2024_assets/shap_out/*.png")


['/workspace/ember2024_assets/shap_out/summary_top30.png',
 '/workspace/ember2024_assets/shap_out/importance_bar_top30.png']

In [17]:
print("X features:", X.shape[1])
print("Model features:", bst.num_feature())


X features: 2568
Model features: 2568


In [18]:
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

# LightGBM predict gives probabilities for binary classification (usually)
p = bst.predict(X)  # shape (n,)

# threshold at 0.5
y_pred = (p >= 0.5).astype(int)

acc = accuracy_score(y, y_pred)
cm = confusion_matrix(y, y_pred)
fpr_tpr_auc = roc_auc_score(y, p)

print("Accuracy:", acc)
print("ROC-AUC:", fpr_tpr_auc)
print("Confusion matrix [ [TN FP], [FN TP] ]:\n", cm)
print("\nClassification report:\n", classification_report(y, y_pred, digits=4))


Accuracy: 0.9803222222222222
ROC-AUC: 0.9981880554115227
Confusion matrix [ [TN FP], [FN TP] ]:
 [[531824   8176]
 [ 13076 526924]]

Classification report:
               precision    recall  f1-score   support

           0     0.9760    0.9849    0.9804    540000
           1     0.9847    0.9758    0.9802    540000

    accuracy                         0.9803   1080000
   macro avg     0.9804    0.9803    0.9803   1080000
weighted avg     0.9804    0.9803    0.9803   1080000



Best threshold (F1): 0.45
Best F1: 0.9804229358359913
Accuracy at best threshold: 0.9804611111111111


In [20]:
import json, os

metrics = {
    "model": "EMBER2024_PE.model",
    "split": "test",
    "n_samples": int(X.shape[0]),
    "n_features": int(X.shape[1]),
    "accuracy": float(acc),
    "roc_auc": float(fpr_tpr_auc),
    "confusion_matrix": cm.tolist(),
}

out_path = "/workspace/ember2024_assets/shap_out/baseline_metrics.json"
with open(out_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved:", out_path)


Saved: /workspace/ember2024_assets/shap_out/baseline_metrics.json
